# 1. ChromData — the core data container

`uchrom.ChromData` is the central data structure for U-Chrom, analogous to
[AnnData](https://anndata.readthedocs.io/) for chromatin 3D structure data.
It holds genomic bins mapped to 3D coordinates, grouped in a
**Cell → Trace → Spot** hierarchy, and persists as `.h5cd` (HDF5) files.

This notebook covers:

- Creating a ChromData from scratch
- The three "core tables" (core / cell / chromatin) and auxiliary storage
- Subsetting by chromosome, trace, or cell
- Round-tripping to `.h5cd` with format versioning
- Importing from reconstruction CSV and FOF-CT


In [1]:
import numpy as np
import pandas as pd
from uchrom import ChromData


## Minimal construction

`ChromData` needs at least `coords (n_spots, 3)` and a `spots` DataFrame with `chrom / start / end / trace_id`.


In [2]:
rng = np.random.default_rng(0)
coords = rng.normal(size=(60, 3))
spots = pd.DataFrame({
    'chrom':    ['chr1']*30 + ['chr2']*30,
    'start':    list(range(0, 30_000_000, 1_000_000)) * 2,
    'end':      list(range(1_000_000, 31_000_000, 1_000_000)) * 2,
    'trace_id': [0]*30 + [1]*30,
})

cd = ChromData(coords, spots, uns={'genome_assembly': 'GRCh38'})
cd


ChromData: n_spots=60, n_traces=2
  spots:   ['chrom', 'start', 'end', 'trace_id']
  uns:     ['genome_assembly']

## Attributes


In [3]:
print('n_spots :', cd.n_spots)
print('n_traces:', cd.n_traces)
print('chroms  :', cd.chroms)
cd.spots.head()


n_spots : 60
n_traces: 2
chroms  : ['chr1', 'chr2']


,chrom,start,end,trace_id
0,chr1,0,1000000,0
1,chr1,1000000,2000000,0
2,chr1,2000000,3000000,0
3,chr1,3000000,4000000,0
4,chr1,4000000,5000000,0


`spots['chrom']` and `spots['trace_id']` are auto-converted to `pd.Categorical` for ~10× memory savings on large datasets.


In [4]:
print(cd.spots['chrom'].dtype)
print(cd.spots['trace_id'].dtype)


category
category


## Subsetting

All subset operations return a **new** `ChromData` — never mutate the original.


In [5]:
chr1 = cd.get_chrom('chr1')
print(chr1)


ChromData: n_spots=30, n_traces=1
  spots:   ['chrom', 'start', 'end', 'trace_id']
  uns:     ['genome_assembly']


In [6]:
trace0 = cd.get_trace(0)
print(trace0)

# Boolean indexing also works
first_ten = cd[np.arange(10)]
print(first_ten)


ChromData: n_spots=30, n_traces=1
  spots:   ['chrom', 'start', 'end', 'trace_id']
  uns:     ['genome_assembly']
ChromData: n_spots=10, n_traces=1
  spots:   ['chrom', 'start', 'end', 'trace_id']
  uns:     ['genome_assembly']


## On-demand distance matrix

Global pairwise distance matrices are not stored (they'd be O(n²) and biologically meaningless across cells). Compute per-trace on demand:


In [7]:
d = cd.compute_distances(trace_id=0)
print('distance matrix shape:', d.shape)
print('diagonal is zero      :', np.allclose(np.diag(d), 0))


distance matrix shape: (30, 30)
diagonal is zero      : True


## Persistence — `.h5cd`

`ChromData` serialises to HDF5 with a format version attribute so future releases can migrate old files.


In [8]:
import tempfile, os
tmp = tempfile.mkdtemp()
path = os.path.join(tmp, 'demo.h5cd')

cd.write(path)
cd2 = ChromData.read(path)
print(cd2)
print('coords match:', np.allclose(cd.coords, cd2.coords))


ChromData: n_spots=60, n_traces=2
  spots:   ['chrom', 'start', 'end', 'trace_id']
  uns:     ['genome_assembly']
coords match: True


## Importing from existing formats

| Source | Method |
|--------|--------|
| Reconstruction CSV (chrom, start, end, x, y, z) | `ChromData.from_dataframe(df)` |
| 4DN FOF-CT core table | `ChromData.from_fofct(path)` |
| `.h5cd` | `ChromData.read(path)` |


In [9]:
# Example: from a reconstruction output CSV (here we build a fake one)
recon = pd.DataFrame({
    'chrom': ['chr1']*10,
    'start': range(0, 10_000_000, 1_000_000),
    'end':   range(1_000_000, 11_000_000, 1_000_000),
    'x': rng.normal(size=10),
    'y': rng.normal(size=10),
    'z': rng.normal(size=10),
})
cd_recon = ChromData.from_dataframe(recon, cell_id='demo_cell')
cd_recon


ChromData: n_spots=10, n_traces=1, n_cells=1
  spots:   ['chrom', 'start', 'end', 'trace_id', 'cell_id']

## Where to go next

- **Tutorial 2** — importing real FOF-CT imaging data and inspecting trace structure.
- **Tutorial 3** — running 3D reconstruction from Hi-C `.pairs` files.
- **Tutorial 4** — loop calling on FOF-CT data.
